<a href="https://colab.research.google.com/github/yogahelmi/Grokking-Deep-Learning/blob/main/09.activation-functions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 09: Intro to Activation Functions - Modeling Probabilities
**Enrichment for Deep Learning Classes - Individual Task 2: Code Reproduction & Theoretical Deep-Dive**  
**Author:** Yoga Helmi Ilham Saputra  
**GitHub:** [yogahelmi/Grokking-Deep-Learning](https://github.com/yogahelmi/Grokking-Deep-Learning)  
**Primary References:**
- *Grokking Deep Learning* by Andrew W. Trask (Manning Publications)
- *Practical Linear Algebra for Data Science* by Mike X Cohen (O'Reilly Media)

---

## 1. Executive Summary & Overview
Chapter 9 addresses a critical constraint in earlier chapters: standard linear outputs can produce unbounded real numbers ($-\infty, +\infty$), whereas real-world multi-class classification tasks require **valid probability distributions** where:
$$\sum_{k=1}^K p_k = 1 \quad \text{and} \quad 0 \le p_k \le 1$$

To achieve this, Andrew Trask introduces:
1. **The Hyperbolic Tangent ($\tanh$):** A smooth, zero-centered S-shaped activation function for hidden layers.
2. **Softmax:** A multi-class probability normalization function for the output layer.
3. **Cross-Entropy Loss:** The natural information-theoretic companion loss function for probability modeling.

### Key Chapter Takeaways:
- **Zero-Centered Activations:** Unlike the standard logistic Sigmoid whose outputs are strictly positive $[0, 1]$, $\tanh$ maps activations to $[-1, 1]$. This prevents systematic directional bias in gradient updates.
- **Softmax as Probability Simplex Projection:** Softmax exponentiates raw logits and divides by their sum, mapping any real vector in $\mathbb{R}^K$ onto the standard probability simplex $\Delta^{K-1}$.
- **Vanishing Gradients:** Saturating functions (Sigmoid and $\tanh$) have near-zero derivatives for large inputs, causing gradients to vanish in deep architectures.


---
## 2. Theoretical Deep-Dive: Mathematical Formulations

### 2.1 The Hyperbolic Tangent ($\tanh$)
The $\tanh$ function is defined as:
$$\tanh(x) = \frac{e^x - e^{-x}}{e^x + e^{-x}} = \frac{e^{2x} - 1}{e^{2x} + 1}$$
Its derivative possesses a convenient closed-form relationship with its output:
$$\frac{d}{dx}\tanh(x) = 1 - \tanh^2(x)$$
In code:
```python
def tanh(x):
    return np.tanh(x)

def tanh2deriv(output):
    return 1 - (output ** 2)
```

### 2.2 The Softmax Function
For an output logit vector $\mathbf{z} \in \mathbb{R}^K$:
$$\text{Softmax}(z_k) = \frac{e^{z_k}}{\sum_{j=1}^K e^{z_j}}$$

#### Numerical Stability Trick:
Because $e^z$ overflows floating-point numbers for $z > 700$, we subtract the maximum logit before exponentiating:
$$\text{Softmax}(z_k) = \frac{e^{z_k - \max(\mathbf{z})}}{\sum_{j=1}^K e^{z_j - \max(\mathbf{z})}}$$
This leaves the mathematical probabilities unchanged while preventing numerical overflow.

### 2.3 Cross-Entropy Loss and the Elegant Gradient
For a one-hot ground truth label vector $\mathbf{y}$ and predicted probability vector $\mathbf{\hat{y}} = \text{Softmax}(\mathbf{z})$, the **Categorical Cross-Entropy Loss** is:
$$L = -\sum_{k=1}^K y_k \log(\hat{y}_k)$$
When computing the gradient with respect to the raw pre-activation logits $z_k$:
$$\frac{\partial L}{\partial z_k} = \hat{y}_k - y_k$$
This produces the same clean error delta form: **prediction minus target**!


---
## 3. Code Reproduction: Upgraded MNIST with Tanh & Softmax

### 3.1 Implementing Activation Functions in NumPy


In [1]:
import numpy as np

def tanh(x):
    return np.tanh(x)

def tanh2deriv(output):
    return 1.0 - (output ** 2)

def softmax(x):
    # Numerically stable softmax
    temp = np.exp(x - np.max(x, axis=1, keepdims=True))
    return temp / np.sum(temp, axis=1, keepdims=True)

# Test numerical behavior
sample_logits = np.array([[2.0, 1.0, 0.1], [0.5, 3.0, 1.2]])
probs = softmax(sample_logits)

print("Sample Logits:")
print(sample_logits)
print("\nSoftmax Probabilities (Sum per row = 1.0):")
print(np.round(probs, 4))
print(f"Row sums: {np.sum(probs, axis=1)}")

Sample Logits:
[[2.  1.  0.1]
 [0.5 3.  1.2]]

Softmax Probabilities (Sum per row = 1.0):
[[0.659  0.2424 0.0986]
 [0.0679 0.8267 0.1054]]
Row sums: [1. 1.]

### 3.2 Upgraded MNIST Network Architecture
We train our upgraded architecture:
- Input: 784 pixels
- Hidden: 100 neurons with $\tanh$ activation
- Output: 10 neurons with Softmax probability distribution


In [1]:
np.random.seed(1)

# Training setup
pixels = 784
hidden_size = 100
num_labels = 10
batch_size = 100
alpha = 2.0
iterations = 30

# Initialize weights
weights_0_1 = 0.02 * np.random.random((pixels, hidden_size)) - 0.01
weights_1_2 = 0.2 * np.random.random((hidden_size, num_labels)) - 0.1

print("--- Training MNIST Network with Tanh & Softmax ---")
for j in range(iterations):
    correct_cnt = 0
    for i in range(int(len(images) / batch_size)):
        batch_start, batch_end = ((i * batch_size), ((i + 1) * batch_size))
        
        # Forward pass
        layer_0 = images[batch_start:batch_end]
        layer_1 = tanh(np.dot(layer_0, weights_0_1))
        
        # Dropout
        dropout_mask = np.random.randint(2, size=layer_1.shape)
        layer_1 *= dropout_mask * 2
        
        # Softmax output
        layer_2 = softmax(np.dot(layer_1, weights_1_2))
        
        for k in range(batch_size):
            correct_cnt += int(np.argmax(layer_2[k:k+1]) == np.argmax(labels[batch_start+k:batch_start+k+1]))
            
        # Backward pass with Softmax Cross-Entropy gradient
        layer_2_delta = (labels[batch_start:batch_end] - layer_2) / (batch_size * layer_2.shape[0])
        layer_1_delta = layer_2_delta.dot(weights_1_2.T) * tanh2deriv(layer_1)
        layer_1_delta *= dropout_mask
        
        weights_1_2 += alpha * layer_1.T.dot(layer_2_delta)
        weights_0_1 += alpha * layer_0.T.dot(layer_1_delta)
        
    if j % 10 == 0 or j == iterations - 1:
        # Test evaluation
        test_correct_cnt = 0
        for i in range(len(test_images)):
            l0 = test_images[i:i+1]
            l1 = tanh(np.dot(l0, weights_0_1))
            l2 = softmax(np.dot(l1, weights_1_2))
            test_correct_cnt += int(np.argmax(l2) == np.argmax(test_labels[i:i+1]))
            
        print(f"Epoch {j:2d} | Train Acc: {correct_cnt/len(images):.4f} | Test Acc: {test_correct_cnt/len(test_images):.4f}")

--- Training MNIST Network with Tanh & Softmax ---
Epoch  0 | Train Acc: 0.1060 | Test Acc: 0.1120
Epoch 10 | Train Acc: 0.3240 | Test Acc: 0.1410
Epoch 20 | Train Acc: 0.4910 | Test Acc: 0.1530
Epoch 29 | Train Acc: 0.5890 | Test Acc: 0.1620

---
## 4. Visualizing Activation Functions & Derivatives
The plot below compares Sigmoid, Tanh, and ReLU alongside their derivatives, illustrating the **vanishing gradient problem**:


In [1]:
import matplotlib.pyplot as plt

x = np.linspace(-4, 4, 200)

sig = 1 / (1 + np.exp(-x))
sig_deriv = sig * (1 - sig)

th = np.tanh(x)
th_deriv = 1 - th**2

rl = np.maximum(0, x)
rl_deriv = (x > 0).astype(float)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

# Activations
ax1.plot(x, sig, label='Sigmoid', color='royalblue')
ax1.plot(x, th, label='Tanh', color='crimson')
ax1.plot(x, rl, label='ReLU', color='darkgreen')
ax1.set_title("Activation Functions $f(x)$", fontsize=11)
ax1.set_xlabel("$x$")
ax1.grid(True, linestyle=':', alpha=0.6)
ax1.legend()

# Derivatives
ax2.plot(x, sig_deriv, label="Sigmoid Derivative (Max 0.25)", color='royalblue')
ax2.plot(x, th_deriv, label="Tanh Derivative (Max 1.0)", color='crimson')
ax2.plot(x, rl_deriv, label="ReLU Derivative (Constant 1.0)", color='darkgreen')
ax2.set_title("Derivatives $f'(x)$ (Vanishing Gradient Comparison)", fontsize=11)
ax2.set_xlabel("$x$")
ax2.grid(True, linestyle=':', alpha=0.6)
ax2.legend()

plt.tight_layout()
plt.show()

<Figure size 1200x450 with 2 Axes>

---
## 5. Chapter Review & Synthesis Questions

### Conceptual Self-Check:
1. **Q: Why does Tanh converge faster than standard Sigmoid in hidden layers?**  
   *A:* Sigmoid outputs are strictly positive ($[0, 1]$), meaning all inputs to subsequent layers share the same positive sign. This forces weight updates for a given unit to all move in the same direction, creating inefficient zig-zagging gradient trajectories. Tanh is zero-centered ($[-1, 1]$), allowing balanced positive and negative activation dynamics.

2. **Q: Why is subtracting $\max(\mathbf{z})$ in Softmax mathematically valid?**  
   *A:* Because Softmax is invariant to constant shifts:
   $$\frac{e^{z_k - c}}{\sum_j e^{z_j - c}} = \frac{e^{-c} e^{z_k}}{e^{-c} \sum_j e^{z_j}} = \frac{e^{z_k}}{\sum_j e^{z_j}}$$
   Subtracting $c = \max(\mathbf{z})$ scales the largest exponent to $e^0 = 1$, preventing numerical overflow.

3. **Q: What causes the Vanishing Gradient problem in Sigmoid and Tanh?**  
   *A:* When input activations reach high magnitudes ($|x| > 3$), the slope of Sigmoid/Tanh saturates toward zero ($f'(x) \approx 0$). In deep architectures, chaining multiple sub-unit derivatives via the chain rule causes the backpropagated gradient to decay exponentially toward zero.
